In [1]:
# ==============================================================================
# DISEASE PREDICTION MODEL -> TFLITE EXPORT PIPELINE
# (Replaces the sklearn ensemble with a Keras model so it can be converted
#  to .tflite for the Flutter app. sklearn/XGBoost models CANNOT be
#  converted to TFLite directly — this is why the app was failing to load.)
# ==============================================================================

import pandas as pd
import numpy as np
import json
import os
import glob

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# ------------------------------------------------------------------------------
# 1. LOAD DATASETS
# ------------------------------------------------------------------------------

def find_file(filename):
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if not matches:
        all_files = glob.glob('/kaggle/input/**/*', recursive=True)
        for f in all_files:
            if filename.lower() in os.path.basename(f).lower():
                return f
        raise FileNotFoundError(f"Could not find {filename} in /kaggle/input/")
    return matches[0]

dataset_path = find_file('DiseaseAndSymptoms.csv') if glob.glob('/kaggle/input/**/DiseaseAndSymptoms.csv', recursive=True) else find_file('dataset.csv')
precaution_path = find_file('Disease precaution.csv') if glob.glob('/kaggle/input/**/Disease precaution.csv', recursive=True) else find_file('symptom_precaution.csv')

print(f"Dataset path found: {dataset_path}")
print(f"Precaution path found: {precaution_path}")

symptoms_df = pd.read_csv(dataset_path)
precautions_df = pd.read_csv(precaution_path)

# ------------------------------------------------------------------------------
# 2. FEATURE EXTRACTION & ONE-HOT ENCODING
# ------------------------------------------------------------------------------
print("Preprocessing symptoms into binary feature vectors...")

symptom_cols = [f'Symptom_{i}' for i in range(1, 18) if f'Symptom_{i}' in symptoms_df.columns]
all_symptoms = set()

for col in symptom_cols:
    unique_in_col = [str(s).strip() for s in symptoms_df[col].dropna().unique()]
    all_symptoms.update(unique_in_col)

symptom_list = sorted(list(all_symptoms))
print(f"Total Unique Diseases: {symptoms_df['Disease'].nunique()}")
print(f"Total Unique Symptoms Extracted: {len(symptom_list)}")

encoded_rows = []
for idx, row in symptoms_df.iterrows():
    present_symptoms = set(str(s).strip() for s in row[symptom_cols].dropna().values)
    row_dict = {symptom: (1 if symptom in present_symptoms else 0) for symptom in symptom_list}
    row_dict['Disease'] = row['Disease']
    encoded_rows.append(row_dict)

dataset_processed = pd.DataFrame(encoded_rows).drop_duplicates().reset_index(drop=True)
print(f"Rows after dedup: {len(dataset_processed)}")

X = dataset_processed.drop(columns=['Disease']).values.astype('float32')
y_raw = dataset_processed['Disease']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)
num_features = X.shape[1]

class_counts = pd.Series(y).value_counts()
use_stratify = class_counts.min() >= 2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if use_stratify else None
)

# ------------------------------------------------------------------------------
# 3. BUILD & TRAIN A KERAS MODEL (this is what actually converts to TFLite)
# ------------------------------------------------------------------------------
print(f"\nBuilding Keras model: {num_features} inputs -> {num_classes} disease classes")

model = keras.Sequential([
    keras.layers.Input(shape=(num_features,)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=60,
    batch_size=16,
    verbose=2
)

y_pred = np.argmax(model.predict(X_test), axis=1)
acc = accuracy_score(y_test, y_pred)
print(f"\n==========================================")
print(f" KERAS MODEL TEST ACCURACY: {acc * 100:.2f}%")
print(f"==========================================\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))

# ------------------------------------------------------------------------------
# 4. CONVERT TO TFLITE (this is the file the Flutter app actually loads)
# ------------------------------------------------------------------------------
print("Converting to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
# No optimizations — quantization bumps FULLY_CONNECTED to a version
# the tflite_flutter runtime doesn't support yet, causing "Registration failed"
tflite_model = converter.convert()

with open('disease_model.tflite', 'wb') as f:
    f.write(tflite_model)



print(f"Saved disease_model.tflite ({len(tflite_model) / 1024:.1f} KB)")
print(f"Input shape the app must send: [1, {num_features}] float32")
print(f"Output shape the app will receive: [1, {num_classes}] float32 (softmax probabilities)")

# ------------------------------------------------------------------------------
# 5. EXPORT SCHEMA & PRECAUTIONS (unchanged format, still needed by the app)
# ------------------------------------------------------------------------------
symptom_mapping = {
    "symptoms": symptom_list,
    "diseases": list(label_encoder.classes_)
}
with open('symptom_schema.json', 'w') as f:
    json.dump(symptom_mapping, f, indent=4)

precaution_cols = [c for c in precautions_df.columns if 'precaution' in c.lower()]
precaution_map = {}
for _, row in precautions_df.iterrows():
    d_name = row['Disease']
    p_list = [str(row[c]).title() for c in precaution_cols if pd.notna(row[c]) and str(row[c]) != 'nan']
    precaution_map[d_name] = p_list

with open('disease_precautions.json', 'w') as f:
    json.dump(precaution_map, f, indent=4)

print("\nAll assets saved:")
print(" - disease_model.tflite      <- copy to flutter_app/assets/")
print(" - symptom_schema.json       <- copy to flutter_app/assets/")
print(" - disease_precautions.json  <- copy to flutter_app/assets/")
print("\nNOTE: Every disease name in symptom_schema.json['diseases'] should match")
print("a key in disease_precautions.json. Any mismatch just means that disease")
print("won't have precautions shown (app already handles this gracefully).")

Dataset path found: /kaggle/input/datasets/choongqianzheng/disease-and-symptoms-dataset/DiseaseAndSymptoms.csv
Precaution path found: /kaggle/input/datasets/choongqianzheng/disease-and-symptoms-dataset/Disease precaution.csv
Preprocessing symptoms into binary feature vectors...
Total Unique Diseases: 41
Total Unique Symptoms Extracted: 131
Rows after dedup: 304

Building Keras model: 131 inputs -> 41 disease classes


I0000 00:00:1787229214.380172      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787229214.383017      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        33,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 41)             │         5,289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,977 (281.16 KB)

...

16/16 - 5s - 325ms/step - accuracy: 0.1070 - loss: 3.6356 - val_accuracy: 0.4754 - val_loss: 3.4149
Epoch 2/60
16/16 - 0s - 9ms/step - accuracy: 0.4403 - loss: 3.2859 - val_accuracy: 0.7049 - val_loss: 2.9711
Epoch 3/60
16/16 - 0s - 9ms/step - accuracy: 0.5885 - loss: 2.8299 - val_accuracy: 0.7541 - val_loss: 2.3577
Epoch 4/60
16/16 - 0s - 10ms/step - accuracy: 0.6749 - loss: 2.2097 - val_accuracy: 0.8852 - val_loss: 1.6976
Epoch 5/60
16/16 - 0s - 9ms/step - accuracy: 0.8025 - loss: 1.6614 - val_accuracy: 0.9508 - val_loss: 1.1606
Epoch 6/60
16/16 - 0s - 11ms/step - accuracy: 0.9259 - loss: 1.1403 - val_accuracy: 0.9836 - val_loss: 0.7668
Epoch 7/60
16/16 - 0s - 9ms/step - accuracy: 0.9383 - loss: 0.8413 - val_accuracy: 0.9836 - val_loss: 0.5131
Epoch 8/60
16/16 - 0s - 9ms/step - accuracy: 0.9794 - loss: 0.5722 - val_accuracy: 1.0000 - val_loss: 0.3379
Epoch 9/60
16/16 - 0s - 9ms/step - accuracy: 0.9918 - loss: 0.3882 - val_accuracy: 1.0000 - val_loss: 0.2157
Epoch 10/60
16/16 - 0s - 9

INFO:tensorflow:Assets written to: /tmp/tmpfauqu5tv/assets


Saved artifact at '/tmp/tmpfauqu5tv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 131), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 41), dtype=tf.float32, name=None)
Captures:
  138687235102608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138684530143120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138684530144464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138684530142736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138684530142544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138684530144656: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1787229230.574843      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1787229230.574866      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1787229230.579614      24 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


Saved disease_model.tflite (283.2 KB)
Input shape the app must send: [1, 131] float32
Output shape the app will receive: [1, 41] float32 (softmax probabilities)

All assets saved:
 - disease_model.tflite      <- copy to flutter_app/assets/
 - symptom_schema.json       <- copy to flutter_app/assets/
 - disease_precautions.json  <- copy to flutter_app/assets/

NOTE: Every disease name in symptom_schema.json['diseases'] should match
a key in disease_precautions.json. Any mismatch just means that disease
won't have precautions shown (app already handles this gracefully).


In [2]:
# ==============================================================================
# ZIP MODEL TRAINING OUTPUTS FOR DOWNLOAD
# Run this AFTER train_and_export_tflite.py has already run in this notebook
# session. Packages outputs into the repo's model_training/outputs/ structure.
# ==============================================================================

import os
import shutil
import zipfile

REPO_NAME = "symptom-checker-tflite-flutter"

output_files = [
    "disease_model.tflite",
    "symptom_schema.json",
    "disease_precautions.json",
]

# ------------------------------------------------------------------------------
# 1. BUILD FOLDER STRUCTURE
# ------------------------------------------------------------------------------
if os.path.exists(REPO_NAME):
    shutil.rmtree(REPO_NAME)

base = os.path.join(REPO_NAME, "model_training", "outputs")
os.makedirs(base)

missing = []
for fname in output_files:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(base, fname))
    else:
        missing.append(fname)

if missing:
    print(f"WARNING: these expected files were not found and were skipped: {missing}")
    print("Make sure train_and_export_tflite.py ran successfully in this session first.")

# ------------------------------------------------------------------------------
# 2. ZIP IT
# ------------------------------------------------------------------------------
zip_path = f"{REPO_NAME}.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(REPO_NAME):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start='.')
            zf.write(file_path, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\nCreated {zip_path} ({zip_size_mb:.2f} MB)")
print("Contents:")
with zipfile.ZipFile(zip_path, 'r') as zf:
    for name in zf.namelist():
        print(f"  {name}")

print(f"\nGo to the 'Output' tab on the right side of the Kaggle notebook,")
print(f"find {zip_path}, and click the download icon.")
print(f"\nAfter unzipping, these three files ALSO need to be copied into")
print(f"flutter_app/assets/ (same three files, both locations):")
for f in output_files:
    print(f"  - {f}")
    


Created symptom-checker-tflite-flutter.zip (0.26 MB)
Contents:
  symptom-checker-tflite-flutter/model_training/outputs/disease_precautions.json
  symptom-checker-tflite-flutter/model_training/outputs/symptom_schema.json
  symptom-checker-tflite-flutter/model_training/outputs/disease_model.tflite

Go to the 'Output' tab on the right side of the Kaggle notebook,
find symptom-checker-tflite-flutter.zip, and click the download icon.

After unzipping, these three files ALSO need to be copied into
flutter_app/assets/ (same three files, both locations):
  - disease_model.tflite
  - symptom_schema.json
  - disease_precautions.json
